_____________
## **Previsão de Risco de Perda de OLA por Incidente**

Modelo de Classificação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_excel('LW-DATASET-TRATADO.xlsx')
df.shape

##### Estamos tratando de uma variável **target extremamente desbalanceada**

Por isso, nestes caso, eu gosto de usar undersampling e class_weight=balanced (nos modelos que permitem)

In [ ]:
df['kpi_violado'].value_counts()

In [ ]:
df_kpi = df.copy()

Para o treino deste modelo existe uma série de features que DEVEM FICAR DE FORA (pois configuram data leak) e outras features que não deveriam entrar por não agregarem muito ao modelo (vou filtrar e elas vão se tornar inúteis):
- Primeiramente vou filtrar o dataset por 'Entrou para KPI?'] == 'SIM', filtrando assim algumas features se tornariam inúteis:
    - kpi_nao_aplicavel
    - tem_pai --> as que tem não entram pra kpi
    - 'Entrou para KPI?' --> usei só para filtrar
    - 'Incidente Pai' --> se tem pai, não entra no kpi

Além disso, existem features que não agregam valor por si próprias:
- 'Número' --> id, sem valor preditivo


In [ ]:
df_kpi = df[df['Entrou para KPI?'] == 'SIM'].copy()
y = df_kpi['kpi_violado']

In [ ]:
colunas_remover = [
    # leak
    'Resolvido','Encerrado','Duração','Código de fechamento',
    'Solução','Status','tem_resolucao','tem_CF',
    # não úteis por conta do filtro
    'KPI Violado?','Entrou para KPI?','kpi_nao_aplicavel','Incidente Pai','tem_pai',
    # sem valor
    'Número',
    # texto livre (sem NLP)
    'Descrição resumida']

X = df_kpi.drop(columns=colunas_remover)

X['hora_abertura'] = df_kpi['Aberto'].dt.hour
X['dia_semana']    = df_kpi['Aberto'].dt.dayofweek
X['fim_de_semana'] = (df_kpi['Aberto'].dt.dayofweek >= 5).astype(int)
X = X.drop(columns=['Aberto'])

In [ ]:
matriz_correlacao = X.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(25, 10))
sns.heatmap(matriz_correlacao, annot = True, linewidths = 0.2, cmap='coolwarm', fmt='.2f')
plt.show()

In [ ]:
X.drop(columns=['kpi_violado'], inplace=True)

In [ ]:
X.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

In [ ]:
colunas_dummy = [
    'Prioridade', 'Produto', 'Categoria', 'Subcategoria',
    'Grupo designado', 'Item de configuração', 'Aberto por']

X_train = pd.get_dummies(X_train, columns=colunas_dummy, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=colunas_dummy, drop_first=True)

# alinhando as colunas de treino e teste
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

``Undersampling``

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("\nAntes do undersampling:")
print(y_train.value_counts())
print("\nDepois do undersampling:")
print(y_train_under.value_counts())

### Testando Modelos 
Logistic Regression, Random Forest, XGBoost e Gradient Boosting

``IMPORTANTE:``

O undersampling foi feito nos dados de treino, e não foi colocado nos dados de teste JUSTAMENTE para manter a ideia de cenário real "desbalanceado" na hora de avaliar o modelo.
Assim, eu mantenho a integridade do modelo, treinando ele para entender os casos, mas sem deixar inutilizável em contexto real

In [ ]:
from sklearn.metrics import precision_score, accuracy_score, confusion_matrix, roc_auc_score, recall_score, classification_report

def avaliar_modelo(y_test, y_pred):
    print(f"acuracia: {accuracy_score(y_test, y_pred)}")
    print(f"precisao: {precision_score(y_test, y_pred)}")
    print(f"recall: {recall_score(y_test, y_pred)}")
    cm = confusion_matrix(y_test, y_pred)
    print(f"roc auc: {roc_auc_score(y_test, y_pred)}\n")

    print(classification_report(y_test, y_pred))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.show()

``Logistic Regression``

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=10000, class_weight='balanced')

model.fit(X_train_under, y_train_under)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

avaliar_modelo(y_test, y_pred)

``RandomForestClassifier``

A acurácia de 99% parece bonita, mas o recall tá extremamente baixo

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    class_weight='balanced')

rf.fit(X_train_under, y_train_under)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

threshold = 0.25
y_pred_rf = (y_proba_rf >= threshold).astype(int)

avaliar_modelo(y_test, y_pred_rf)

``XGBClassifier``

In [ ]:
from xgboost import XGBClassifier

scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42, 
    scale_pos_weight=scale_pos_weight)

xgb.fit(X_train_under, y_train_under)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

avaliar_modelo(y_test, y_pred_xgb)

``GradientBoostingClassifier``

Threshold 0.5: 
- acurácia 0.83
- recall 0.68
- precisão 0.04
- roc 0.75

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42)

gb.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_gb = gb.predict(X_test)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

avaliar_modelo(y_test, y_pred_gb)

### **Testando Thresholds**

Trade off entre acurácia e recall

Como o meu caso o mais importante é **detectar a maioria de casos que vão violar**, o ``recall é o mais importante`` para mim

In [ ]:
for threshold in [0.2, 0.3, 0.4]:    
    y_pred = (y_proba_gb >= threshold).astype(int)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred)

    print('threshold:', threshold)
    avaliar_modelo(y_test, y_pred)

Para o meu caso, decidi escolher threshold 0.3
- recall 0.94 (extremamente interessante para o objetivo deste modelo)
- acurácia 0.58
- roc auc 0.75

In [ ]:
print(classification_report(y_test, y_pred_gb))

Modelo final indicando Probabilidades

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    random_state=42)

gb.fit(X_train, y_train, sample_weight=sample_weights)

# extraindo probabilidades da classe 1 (KPI violado)
y_proba_gb = gb.predict_proba(X_test)[:, 1]

threshold = 0.3
y_pred_gb_custom = (y_proba_gb >= threshold).astype(int)
avaliar_modelo(y_test, y_pred_gb_custom)

In [ ]:
df_resultados_gb = X_test.copy()
df_resultados_gb['KPI_Violado_Real'] = y_test
df_resultados_gb['Previsao_Threshold_0.3'] = y_pred_gb_custom
df_resultados_gb['Probabilidade_Violacao'] = y_proba_gb
df_resultados_gb['Probabilidade_Violacao_%'] = (y_proba_gb * 100).round(2)

# 15 casos com maior probabilidade de estouro do KPI
display(df_resultados_gb.sort_values(by='Probabilidade_Violacao', ascending=False).head(15))

Salvando CSV para PowerBI

In [ ]:
df_export_powerbi = pd.DataFrame()
df_export_powerbi['Numero_Incidente'] = df_kpi.loc[X_test.index, 'Número']

df_export_powerbi['Probabilidade_Violacao'] = y_proba_gb
df_export_powerbi['Probabilidade_Violacao_%'] = (y_proba_gb * 100).round(2)

df_export_powerbi = df_export_powerbi.sort_values(by='Probabilidade_Violacao', ascending=False)

filename = "export_previsoes_kpi.csv"
df_export_powerbi.to_csv(filename, index=False)

print(f"Arquivo '{filename}' gerado com sucesso!")